# Integrantes


## Cosas a Abstraer
- Mapa (Tablero)
(debemos crearlos manualmente o hacer una funcion que pueda crear puzles aleatorios)

- Celdas

 - validaciones clave:
    - Solo se pueden formar palabras de manera horizontal o vertical
    - los espacios extras solo se pueden poner de manera adjacente y horizontal o vertical
    - los espacios ya marcados pasan a no existir, y no afectar.

- Combinaciones de letras especiales

    LOK:
        permite agregar un punto en cualquier celda

    TLAK
        permite agregar dos puntos adjacentes en cualquier celda

    TA:
      Marca todas las celdas que contengan ese mismo valor

    X:
      Es un comodin que permite eliminar dicha casillas

  (no he llegado a ninguna parte del juego donde pueda usar eso)

- Jugador:
    Donde se guardara temporalmente las casillas Seleccionadas



# Condiciones
## Condiciones Basicas (condiciones basicas de posicion)
- Solo se puede elegir en horizontal o vertical
- no se pueden convinar celdas con valores y celdas vacias
- No se puede hacer puente entre casillas vacias sin sellar

## Condiciones Avanzadas (cada cosa que hace cada palabra)
- LOK: cuando la cache tenga un LOK, se sellaran esas celdas y se aplicara, un comodin que sella cualquier casilla donde se ponga
- TLAK: cuando la cache tenga un TLAK, se sellaran esas celdas y se aplicara un comodin que sella dos casillas adjacentes ya sea horizontal o verticalmente, en cualqueir parte del tablero y se debe aplicar justo despues de conseguir el TLAK.
- TA: cuando la cache tenga un TA, se sellaran estas casillas, y se aplicara un comodin el cual sellara todas las casillas que tengan el valor de la casilla isSeleccionada
- X: es un comodin que sella una casilla isSeleccionada.

## Condicion WIN
todas las casillas en sellado

# Imports

In [ ]:
from IPython.display import clear_output, display
import ipywidgets as widgets
import numpy as np

# Backend

In [ ]:
class Mapa:
  def __init__(self, celdas):
    self.celdas = celdas
    self.is_juego_continua = True
    self.mapa = []
    self.crear_mapa()

  def crear_mapa(self):
    if not self.celdas:
      self.mapa = []
      return
    max_fila = max(celda.fila for celda in self.celdas)
    max_columna = max(celda.columna for celda in self.celdas)
    self.mapa = [[None for _ in range(max_columna + 1)] for _ in range(max_fila + 1)]

    # Colocamos cada celda
    for celda in self.celdas:
      self.mapa[celda.fila][celda.columna] = celda

  def imprimir_mapa(self):
    for fila in self.mapa:
      print("\n", end="")
      for celda in fila:
        if celda == None:
          print(" ", end=" ")
        else:
          if celda.isSellada == True:
            print(f"{celda.valor}^", end=" ")
          elif celda.isSeleccionada == True:
            print(f"{celda.valor}*", end=" ")
          else:
            print(celda.valor, end=" ")

  def obtener_celdas_adjacentes(self,celda):
    fila = celda.fila
    columna = celda.columna
    total_filas = len(self.mapa)
    total_cols = len(self.mapa[0]) if total_filas > 0 else 0
    celdas_adjacentes = []
    vecinos = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    for vecino in vecinos:
      fila_vecina = fila + vecino[0]
      columna_vecina = columna + vecino[1]
      while 0 <= fila_vecina < total_filas and 0 <= columna_vecina < total_cols:
        if self.mapa[fila_vecina][columna_vecina] != None:
          if self.mapa[fila_vecina][columna_vecina].isSellada == False:
            celdas_adjacentes.append(self.mapa[fila_vecina][columna_vecina])
            break
          else:
            fila_vecina = fila_vecina + vecino[0]
            columna_vecina = columna_vecina + vecino[1]
        else:
          celdas_adjacentes.append(None)
          break
      if (0 <= fila_vecina < total_filas and 0 <= columna_vecina < total_cols) == False:
        celdas_adjacentes.append(None)
    return celdas_adjacentes

  def obtener_celda_adjacentes_en_direccion(self,celdas_cache):
    fila = 0
    columna = 0
    total_filas = len(self.mapa)
    total_cols = len(self.mapa[0]) if total_filas > 0 else 0
    celda_adjacentes = None

    if celdas_cache[0].fila < celdas_cache[1].fila:
      fila = celdas_cache[-1].fila
      fila = fila + 1
      columna = celdas_cache[-1].columna
      is_adjacente_encontrada = False
      while (0 <= fila < total_filas and 0 <= columna < total_cols) and not is_adjacente_encontrada:
        if self.mapa[fila][columna] != None:
          if self.mapa[fila][columna].isSellada == False:
            celda_adjacentes = self.mapa[fila][columna]
            is_adjacente_encontrada = True
          else:
            fila = fila + 1

    elif celdas_cache[0].fila > celdas_cache[1].fila:
      fila = celdas_cache[-1].fila
      fila = fila - 1
      columna = celdas_cache[-1].columna
      is_adjacente_encontrada = False
      while 0 <= fila < total_filas and 0 <= columna < total_cols and not is_adjacente_encontrada:
        if self.mapa[fila][columna] != None:
          if self.mapa[fila][columna].isSellada == False:
            celda_adjacentes = self.mapa[fila][columna]
            is_adjacente_encontrada = True
          else:
            fila = fila - 1

    elif celdas_cache[0].columna < celdas_cache[1].columna:
      fila = celdas_cache[-1].fila
      columna = celdas_cache[-1].columna
      columna = columna + 1
      is_adjacente_encontrada = False
      while 0 <= fila < total_filas and 0 <= columna < total_cols and not is_adjacente_encontrada:
        if self.mapa[fila][columna] != None:
          if self.mapa[fila][columna].isSellada == False:
            celda_adjacentes = self.mapa[fila][columna]
            is_adjacente_encontrada = True
          else:
            columna = columna + 1

    elif celdas_cache[0].columna > celdas_cache[1].columna:
      fila = celdas_cache[-1].fila
      columna = celdas_cache[-1].columna
      columna = columna - 1
      is_adjacente_encontrada = False
      while 0 <= fila < total_filas and 0 <= columna < total_cols and not is_adjacente_encontrada:
        if self.mapa[fila][columna] != None:
          if self.mapa[fila][columna].isSellada == False:
            celda_adjacentes = self.mapa[fila][columna]
            is_adjacente_encontrada = True
          else:
            columna = columna - 1
    return celda_adjacentes

  def seleccionar_celda(self,codernada,cache):
    cache_provicional = cache
    celda = self.mapa[codernada[0]][codernada[1]]
    celdas_adjacentes = []
    # Primera celda a seleccionar (puede ser cualquiera)
    if len(cache_provicional) == 0:
      if celda.isSeleccionada == False:
        celda.isSeleccionada = True
        cache_provicional.append(celda)
    # Adjacencia e ignorar casillas selladas
    elif celda.isSeleccionada == False and celda.isSellada == False:
        # Segunda celda a seleccionar (Define direccion y no puede ser en diagonal)
        if len(cache_provicional) == 1:
          celdas_adjacentes = self.obtener_celdas_adjacentes(cache_provicional[0])
          is_ninguna_seleccionada = True
          for celda_adjacente in celdas_adjacentes:
              if celda_adjacente == celda:
                is_ninguna_seleccionada = False
                celda.isSeleccionada = True
                cache_provicional.append(celda)

          if is_ninguna_seleccionada:
            print("ninguna de las celdas seleccionadas es adjacente a la primera celda seleccionada.")
        # Tercera o mas casilla a seleccionar (ya tiene una direccion obligatoria a seguir)
        else:
          celda_adjacente = self.obtener_celda_adjacentes_en_direccion(cache_provicional)
          if celda_adjacente == None or celda_adjacente != celda:
            print("ninguna de las celdas seleccionadas es adjacente a la primera celda seleccionada.")
          elif celda_adjacente == celda:
            celda.isSeleccionada = True
            cache_provicional.append(celda)
          else:
            print("ninguna de las celdas seleccionadas es adjacente a la primera celda seleccionada.")

    elif celda.isSellada == True:
      print("Celda sellada elija la proxima adjacente la cual no este sellada")
    elif cache_provicional[-1] == celda:
      cache_provicional.pop()
      celda.isSeleccionada = False
    return cache_provicional

  def sellador(self, recibido):
    for celda in recibido[1]:
      celda.isSellada = True
    print(f"Palabra lograda: {recibido[0]}")
    self.imprimir_mapa()

    if recibido[0] == "LOK":
      is_not_seleccionada = True
      while is_not_seleccionada:
        cordenadas = input("Elija una celda para sellar (coordenadas 2 digitos): ")
        x = int(cordenadas[0])
        y = int(cordenadas[1])
        celda = self.mapa[x][y]
        if celda != None and celda.isSellada == False:
          celda.isSellada = True
          is_not_seleccionada = False
        else:
          print( "error-celda seleccionada fuera del tablero de juego o celda ya esta sellada, intente de nuevo")

    elif recibido[0] == "TLAK":
      is_not_seleccionada = True
      while is_not_seleccionada:
        cordenadas = input("Elija una celda para sellar (coordenadas 2 digitos): ")
        x = int(cordenadas[0])
        y = int(cordenadas[1])
        celda = self.mapa[x][y]
        if celda != None and celda.isSellada == False:
          celda.isSellada = True
          celda2 = input("Elija su segunda casilla adjacente a la primera para sellar (coordenadas 2 digitos): ")
          x2 = int(cordenadas[0])
          y2 = int(cordenadas[1])
          celda2 = self.mapa[x2][y2]
          if celda2 != None and celda2.isSellada == False:
            celdas_adjacentes = self.obtener_celdas_adjacentes(celda)
            is_una_de_las_celdas_adjacentes = False
            for celda_adjacente in celdas_adjacentes:
              if celda_adjacente == celda2:
                celda2.isSellada = True
                is_not_seleccionada = False
                is_una_de_las_celdas_adjacentes = True
            if is_una_de_las_celdas_adjacentes == False:
              print("error-las celdas seleccionadas no son adjacentes, intente de nuevo")
          else:
            print("error-celda seleccionada fuera del tablero de juego o celda ya esta sellada, intente de nuevo")
        else:
          print("error-celda seleccionada fuera del tablero de juego o celda ya esta sellada, intente de nuevo")
    elif recibido[0] == "TA":
      is_not_seleccionada = True
      while is_not_seleccionada:
        cordenadas = input("Elija una celda para sellar todas las de ese valor (coordenadas 2 digitos): ")
        x = int(cordenadas[0])
        y = int(cordenadas[1])
        celda = self.mapa[x][y]
        if celda != None and celda.isSellada == False:
          valor_a_sellar = celda.valor
          for celda in self.celdas:
            if celda.valor == valor_a_sellar:
              celda.isSellada = True
          is_not_seleccionada = False
        else:
          print("error-celda seleccionada fuera del tablero de juego o celda ya esta sellada, intente de nuevo")

  def sellar_palabra_base(self, recibido):
    for celda in recibido[1]:
      celda.isSellada = True

  def aplicar_comodin_lok(self, x, y):
    celda = self.mapa[x][y]
    if celda is not None and not celda.isSellada:
      celda.isSellada = True
      return True
    return False

  def aplicar_comodin_ta(self, x, y):
    celda = self.mapa[x][y]
    if celda is not None and not celda.isSellada:
      valor_a_sellar = celda.valor
      for c in self.celdas:
        if c.valor == valor_a_sellar:
          c.isSellada = True
      return True
    return False

  def aplicar_comodin_tlak(self, celda1_coords, celda2_coords):
    c1 = self.mapa[celda1_coords[0]][celda1_coords[1]]
    c2 = self.mapa[celda2_coords[0]][celda2_coords[1]]
    if c1 is not None and c2 is not None and not c1.isSellada and not c2.isSellada:
      adj = self.obtener_celdas_adjacentes(c1)
      if c2 in adj:
        c1.isSellada = True
        c2.isSellada = True
        return True
    return False

  def chequear_tablero(self):
    selladas = []
    for celda in self.celdas:
      selladas.append(celda.isSellada)
    if np.all(selladas):
      print("\nPuzzle Resuelto")
      self.is_juego_continua = False

class celda:
  def __init__(self, fila, columna, valor):
    self.fila = fila
    self.columna = columna
    self.valor = valor
    self.isSeleccionada = False
    self.isSellada = False

class jugador:
    def __init__(self):
        self.cache = []

    def agregar_celda(self, coordenada, mapa):
        self.cache = mapa.seleccionar_celda(coordenada,self.cache)

    def reiniciar_cache(self):
      for celda in self.cache:
        celda.isSeleccionada = False
      self.cache = []

    def chequear_cache(self):
      palabra = ""
      for celda in self.cache:
        palabra = palabra + celda.valor
      if palabra == "LOK":
        return [palabra, self.cache]
      elif palabra == "TLAK":
        return [palabra, self.cache]
      elif palabra == "TA":
        return [palabra, self.cache]
      elif palabra == "X":
        return [palabra, self.cache]
      else:
        return None

    def obtener_valores_celdas(self):
        return [celda.valor for celda in self.cache]

# Bucle Juego
(en consola)

In [ ]:
def main_consola():
  celda1 = celda(0, 0, "L")
  celda2 = celda(0, 1, "O")
  celda3 = celda(0, 2, "K")
  celda4 = celda(1, 1, "[]")
  celdas = [celda1, celda2, celda3, celda4]
  jugador1 = jugador()
  mapa = Mapa(celdas)
  mapa.imprimir_mapa()
  while mapa.is_juego_continua:
    entrada = input(
        "\n Ingrese como dos digitos la coordenada que desea seleccionar: "
    )
    x = int(entrada[0])
    y = int(entrada[1])
    coordenada = (x, y)

    try:
      jugador1.agregar_celda(coordenada, mapa)
      palabra = jugador1.chequear_cache()
      if palabra != None:
        mapa.sellador(palabra)
        jugador1.reiniciar_cache()
    except Exception as e:
      print(
          f"error: {e}: error-celda seleccionada fuera del tablero de juego,"
          " intente de nuevo"
      )

    print(f"cache del jugador: {jugador1.obtener_valores_celdas()}")
    mapa.imprimir_mapa()
    mapa.chequear_tablero()

main_consola()


L O K 
  []   
 Ingrese como dos digitos la coordenada que desea seleccionar: 00
cache del jugador: ['L']

L* O K 
  []   
 Ingrese como dos digitos la coordenada que desea seleccionar: 01
cache del jugador: ['L', 'O']

L* O* K 
  []   
 Ingrese como dos digitos la coordenada que desea seleccionar: 00
cache del jugador: ['L', 'O']

L* O* K 
  []   
 Ingrese como dos digitos la coordenada que desea seleccionar: 02
Palabra lograda: LOK

L^ O^ K^ 
  []   Elija una celda para sellar (coordenadas 2 digitos): 00
error-celda seleccionada fuera del tablero de juego o celda ya esta sellada, intente de nuevo
Elija una celda para sellar (coordenadas 2 digitos): 11
cache del jugador: []

L^ O^ K^ 
  []^   
Puzzle Resuelto


# Frontend

In [ ]:
def main(celdas_iniciales):
  mapa = Mapa(celdas_iniciales)
  jugador1 = jugador()

  comodin_activo = None
  tlak_primera_celda = None

  css_lok = """
    <style>
        .lok-container {
            display: flex; flex-direction: column; align-items: center;
            font-family: 'Helvetica Neue', Arial, sans-serif; background-color: #f8f9fa;
            padding: 25px; border-radius: 15px; max-width: 450px; margin: auto;
            box-shadow: 0px 8px 20px rgba(0,0,0,0.06);
        }
        .lok-header { font-size: 26px; font-weight: 900; letter-spacing: 3px; color: #111; margin-bottom: 5px; }
        .lok-cache { font-size: 18px; font-weight: 700; color: #333; background: #e9ecef; padding: 6px 16px; border-radius: 20px; margin-bottom: 20px; min-height: 32px; }
        .lok-btn-normal { background-color: #ffffff !important; color: #111111 !important; font-weight: bold !important; font-size: 18px !important; border: 2px solid #222222 !important; border-bottom: 5px solid #222222 !important; border-radius: 8px !important; }
        .lok-btn-seleccionada { background-color: #111111 !important; color: #ffffff !important; font-weight: bold !important; font-size: 18px !important; border: 2px solid #000000 !important; border-bottom: 5px solid #000000 !important; border-radius: 8px !important; }
        .lok-btn-sellada { background-color: #e0e0e0 !important; color: #888888 !important; border: 2px dashed #aaaaaa !important; border-radius: 8px !important; text-decoration: line-through; opacity: 0.6; }
        .lok-status { margin-top: 15px; font-size: 14px; font-weight: 600; color: #d90429; min-height: 24px; }
    </style>
    """
  display(widgets.HTML(value=css_lok))

  output_grid = widgets.Output()
  output_cache = widgets.Output()
  output_status = widgets.Output()

  def render_mapa():
    with output_grid:
      clear_output(wait=True)
      filas_vbox = []
      for r, fila_celdas in enumerate(mapa.mapa):
        botones_row = []
        for c, celda_obj in enumerate(fila_celdas):
          if celda_obj is None:
            btn = widgets.Button(
                description="",
                disabled=True,
                layout=widgets.Layout(
                    width="55px", height="55px", margin="3px"
                ),
            )
            btn.style.button_color = "transparent"
            btn.layout.border = "none"
          else:
            style_class = "lok-btn-normal"
            if celda_obj.isSellada:
              style_class = "lok-btn-sellada"
            elif celda_obj.isSeleccionada:
              style_class = "lok-btn-seleccionada"

            btn = widgets.Button(
                description=str(celda_obj.valor),
                disabled=celda_obj.isSellada or not mapa.is_juego_continua,
                layout=widgets.Layout(
                    width="55px", height="55px", margin="3px"
                ),
            )
            btn.add_class(style_class)
            btn.on_click(lambda b, x=r, y=c: on_celda_click(x, y))

          botones_row.append(btn)
        filas_vbox.append(widgets.HBox(botones_row))

      display(widgets.VBox(filas_vbox))

  def render_cache():
    with output_cache:
      clear_output(wait=True)
      valores = jugador1.obtener_valores_celdas()
      texto_cache = "".join(valores) if valores else "..."
      display(
          widgets.HTML(
              value=f"<div class='lok-cache'>CÁCHE: <span>{texto_cache}</span></div>"
          )
      )

  def render_status(msg=""):
    with output_status:
      clear_output(wait=True)
      if not mapa.is_juego_continua:
        display(
            widgets.HTML(
                value="<div class='lok-status' style='color:#2b9348;"
                " font-size:18px;'>Puzzle Resuelto</div>"
            )
        )
      else:
        display(widgets.HTML(value=f"<div class='lok-status'>{msg}</div>"))

  def on_celda_click(x, y):
    nonlocal comodin_activo, tlak_primera_celda
    mensaje_status = ""

    if comodin_activo:
      if comodin_activo in ["LOK", "X"]:
        if mapa.aplicar_comodin_lok(x, y):
          comodin_activo = None
        else:
          mensaje_status = ("error-celda seleccionada fuera del tablero de juego o celda ya esta sellada, intente de nuevo")

      elif comodin_activo == "TA":
        if mapa.aplicar_comodin_ta(x, y):
          comodin_activo = None
        else:
          mensaje_status = ("error-celda seleccionada fuera del tablero de juego o celda ya esta sellada, intente de nuevo")

      elif comodin_activo == "TLAK":
        if tlak_primera_celda is None:
          tlak_primera_celda = (x, y)
          mensaje_status = ("Elija su segunda casilla adjacente a la primera para sellar")
        else:
          if mapa.aplicar_comodin_tlak(tlak_primera_celda, (x, y)):
            comodin_activo = None
            tlak_primera_celda = None
          else:
            tlak_primera_celda = None
            mensaje_status = ("error-las celdas seleccionadas no son adjacentes, intente de nuevo")

    else:
      try:
        jugador1.agregar_celda((x, y), mapa)
        palabra = jugador1.chequear_cache()

        if palabra is not None:
          mapa.sellar_palabra_base(palabra)
          jugador1.reiniciar_cache()
          comodin_activo = palabra[0]

          if comodin_activo in ["LOK", "X"]:
            mensaje_status = "Elija una celda para sellar"
          elif comodin_activo == "TA":
            mensaje_status = ("Elija una celda para sellar todas las de ese valor")
          elif comodin_activo == "TLAK":
            mensaje_status = "Elija una celda para sellar"

      except Exception as e:
        mensaje_status = (f"error: {e}: error-celda seleccionada fuera del tablero de juego, intente de nuevo")

    mapa.chequear_tablero()
    render_mapa()
    render_cache()
    render_status(mensaje_status)

  header = widgets.HTML(value="<div class='lok-header'>JUEGO LOK</div>")
  contenedor = widgets.VBox(
      [header, output_cache, output_grid, output_status]
  )
  contenedor.add_class("lok-container")

  display(contenedor)
  render_mapa()
  render_cache()
  render_status()

#Bucle juego
(en frontend)

### Mapa 1

In [ ]:
# Definimos las celdas del mapa 1
celda1 = celda(0, 0, "L")
celda2 = celda(0, 1, "O")
celda3 = celda(0, 2, "K")
celda4 = celda(1, 1, "[]")
celdas_ejemplo = [celda1, celda2, celda3, celda4]

# Ejecutamos mapa 1
main(celdas_ejemplo)

HTML(value="\n    <style>\n        .lok-container {\n            display: flex; flex-direction: column; align-…


Puzzle Resuelto


### Mapa 2

In [ ]:
# Definimos las celdas del mapa 2
celda(0, 0, "L")
celda(0, 1, "J")
celda(1, 0, "M")
celda(1, 1, "K")
celdas = [celda1, celda2, celda3, celda4]

# Ejecutamos mapa 1
main(celdas_ejemplo)